# Going further with single-cell data: sub-clustering, trajectories and gene sets

**SupBioTech M2 practical · advanced session** · human skin wound healing · scRNA-seq, donor PWH27

In the first practical we went from a raw count matrix to annotated cell types (keratinocytes, fibroblasts, macrophages...). That was a *global* view. Real questions are usually more specific: **inside** a cell type, are there sub-populations? Do cells form a **continuum** (a differentiation path) rather than separate groups? And when a group of genes goes up, **what does it mean biologically**?

This notebook answers these three questions on the **keratinocytes** (the epidermal cells, the ones that must regrow to close the wound):

| Part | Method | The question we ask |
|---|---|---|
| A | **Sub-clustering** | Which states exist inside the keratinocytes? |
| B | **Trajectory / pseudotime** (PAGA, diffusion pseudotime) | Do these states form an ordered path, and which genes change along it? |
| C | **Gene set enrichment** (over-representation and GSEA) | What biological processes do the changing genes represent? |
| D | Take-home | What can we trust, what can we not? |

> **Prerequisite.** You must have run the first notebook (`Analysis_sc.ipynb`) up to the end: we start from its last checkpoint, `adata_sc_annotated_cell_state.h5ad`.

## How to read this notebook

The rhythm is the same as in the first practical (**Concept**, **Run**, **Break it**, **Question**, **Take-home**). A new point of attention: in this session we use methods that are **very easy to run and very easy to over-interpret**. A pretty UMAP, a smooth trajectory or a table of significant pathways always come out, whatever the data. So we will repeatedly ask: *how would I know if this result were an artefact?*

## Part 1 · Setup

> **Concept** · Same logic as before: import libraries, fix the random seed, define paths. Two new libraries: **gseapy** (gene set enrichment: a Python wrapper of Enrichr and GSEA) and `adjusted_rand_score` from **scikit-learn**, a number that measures how similar two clusterings are (we use it in Part A).
>
> The new folder `data/gene_sets` is a small local cache: gene set libraries are downloaded once and stored there (Part C).

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad
import harmonypy
import gseapy as gp
from sklearn.metrics import adjusted_rand_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 4))
sns.set_style("whitegrid")

BASE = '/Users/sergio/Spatiotemporal_roadmap_of_human_skin_wound_healing'
SC_DATA = f'{BASE}/data/GSE241132_RAW'                   # raw data + published metadata (input)
OUT = f'{BASE}/data/processed_data/SupBioTech'           # checkpoints of the first notebook / our results
GENESET_DIR = f'{BASE}/data/gene_sets'                   # local cache of gene set libraries
os.makedirs(OUT, exist_ok=True); os.makedirs(GENESET_DIR, exist_ok=True)

## Part 2 · Loading only the keratinocytes

> **Concept · why do we *restart* from the raw counts?** In the first notebook, normalization, highly variable genes (HVG), PCA and neighbors were computed on **all** cell types together. The HVG list was therefore dominated by the genes that distinguish a fibroblast from a macrophage. Those genes say nothing about the *differences between keratinocytes*. If we simply re-clustered the existing PCA, we would be looking at keratinocytes through a lens built for another question.
>
> The rule is: **when you zoom into a subset, redo the pipeline on the subset, starting from the raw counts** (normalization → HVG → PCA → neighbors → clustering). Only the cell selection is inherited.
>
> Technically: the checkpoint is large (several GB in memory). We open it with `backed="r"` (the file stays on disk, nothing is loaded), select the keratinocytes, and bring **only those** into memory with `.to_memory()`. Then we build a fresh `AnnData` from the raw counts that we saved in `layers["counts"]` in the first notebook.

In [ ]:
big = sc.read_h5ad(f"{OUT}/adata_sc_annotated_cell_state.h5ad", backed="r")
is_kera = (big.obs["cell_type"] == "Keratinocytes").values
sub = big[is_kera].to_memory()
big.file.close()

# a clean object: raw counts in X, only the metadata we need
kera = ad.AnnData(X=sub.layers["counts"].copy(),
                  obs=sub.obs[["sample", "cell_type", "cell_state", "paper_barcode"]].copy(),
                  var=pd.DataFrame(index=sub.var_names))
del sub

# published annotation of the authors (finer than the one of notebook 1), matched by barcode
paper_meta = pd.read_csv(f"{SC_DATA}/GSE241132_cell_metadata.txt", sep="\t", index_col=0)
kera.obs["paper_state"] = kera.obs["paper_barcode"].map(paper_meta["newCellTypes"])

sc.pp.filter_genes(kera, min_cells=10)       # genes seen in fewer than 10 keratinocytes carry no information here
print(kera)
print(kera.obs["sample"].value_counts().sort_index().to_dict())
print(kera.obs["paper_state"].value_counts().to_dict())

Notice the last output: the authors distinguished several keratinocyte **states** (`Bas-I`, `Bas-II`, `Bas-mig`, `Bas-prolif`, `HF`, `Spi-I`, `Spi-II`, `Spi-mig`, `Gra-I`...: basal, hair follicle, spinous, granular, "mig" = migrating/wound-activated). We will **not** use these labels to build our clusters. We will use them at the end as an **external yardstick**: an independent opinion to compare with, not a ground truth.

> **Question** · `kera.X` contains raw integer counts at this point. Why is that important for what comes next?
>
> <details><summary>Show answer</summary>
> Because normalization (`normalize_total` + `log1p`) must be computed on raw counts, and it must be computed again on this subset: the total counts per cell and the scaling are the same as before, but HVG, PCA and neighbors must be recomputed on this subset anyway. Starting from already log-transformed values would mean taking the log twice.
> </details>

## Part A · Sub-clustering the keratinocytes

**Goal.** Find the sub-populations hidden inside the keratinocytes, decide how many there are, and give them a biological name.

**To discuss together:**

- How many sub-clusters would you expect in the epidermis, and why?
- Is a cluster the same as a cell type? What is a *state*?
- How do we choose the resolution of the clustering, if there is no 'right' answer?

### A1 · Normalization, HVG, PCA, neighbors, UMAP

> **Concept** · Same five steps as notebook 1 (normalize to 10 000 counts, `log1p`, 2 000 HVG, 30 PCs, kNN graph, UMAP), now on 11 707 keratinocytes. One difference: the HVG are selected with `batch_key="sample"`, meaning "find the genes that are variable **within each sample**, then combine". This prevents a gene from being called variable only because it differs between D0 and D7 (a batch effect, or true biology, we cannot tell yet).
>
> We save the raw counts in `layers["counts"]` **before** normalizing, so that we can always go back.

In [ ]:
kera.layers["counts"] = kera.X.copy()
sc.pp.normalize_total(kera, target_sum=1e4)
sc.pp.log1p(kera)
sc.pp.highly_variable_genes(kera, n_top_genes=2000, batch_key="sample")
sc.pp.pca(kera, n_comps=30, mask_var="highly_variable", random_state=SEED)
sc.pp.neighbors(kera, n_pcs=30, random_state=SEED)
sc.tl.umap(kera, random_state=SEED)
kera.obsm["X_umap_uncorrected"] = kera.obsm["X_umap"].copy()      # keep it: we will compute another UMAP later
print("HVG:", int(kera.var["highly_variable"].sum()))

### A2 · What does the UMAP show? Biology or batch?

> **Concept** · We colour the same UMAP by three things: the **timepoint** (`sample`), the **authors' state**, and the **keratinocyte layer** from notebook 1. If cells separate by timepoint *within* a single state, then part of what we see is the wound itself (biology) and/or a technical batch effect. In a time course on one donor the two are confounded: D0, D1, D7, D30 are different biopsies processed separately.

In [ ]:
sc.pl.embedding(kera, basis="umap_uncorrected", color=["sample", "paper_state", "cell_state"], ncols=3, wspace=0.4)

> **Question** · If all the D7 cells sit in a separate island, can we conclude that D7 keratinocytes are biologically different?
>
> <details><summary>Show answer</summary>
> Not yet. D7 is a different biopsy, prepared on a different day: a batch effect would produce the same picture. With one donor we cannot separate the two explanations from the UMAP alone. What we can do is check whether the *same cell states* are found at every timepoint (conserved structure), and that is what the next steps do.
> </details>

### A3 · Clustering at three resolutions, without batch correction

> **Concept** · Leiden clustering has one main knob, the **resolution**: low = few large clusters, high = many small ones. There is no 'true' number. We try 0.3, 0.5, 0.8 and compare. We use the `igraph` implementation (`flavor="igraph", directed=False`), faster and the recommended one in scanpy.
>
> To measure whether a clustering is *driven by the batch* we compute the **sample purity**: for each cluster we take the fraction of cells coming from its most represented timepoint, and average over cells. With 4 timepoints, a clustering that ignores the timepoint has purity close to the overall share of the largest sample; a clustering that *is* the timepoint has purity 1.

In [ ]:
for r in [0.3, 0.5, 0.8]:
    sc.tl.leiden(kera, resolution=r, key_added=f"unc_r{r}", flavor="igraph", directed=False, n_iterations=2, random_state=SEED)
    print(r, "->", kera.obs[f"unc_r{r}"].nunique(), "clusters")

In [ ]:
def sample_purity(labels, samples):
    t = pd.crosstab(labels, samples)
    return float(t.max(axis=1).sum() / t.values.sum())      # fraction of cells in the dominant sample of their cluster

def paper_ari(labels):
    ok = kera.obs["paper_state"].notna()
    return float(adjusted_rand_score(kera.obs.loc[ok, "paper_state"], labels[ok]))

def summarize(prefix, resolutions=(0.3, 0.5, 0.8)):
    rows = []
    for r in resolutions:
        lab = kera.obs[f"{prefix}_r{r}"]
        rows.append(dict(clustering=f"{prefix}_r{r}", n_clusters=lab.nunique(),
                         sample_purity=round(sample_purity(lab, kera.obs["sample"]), 3),
                         ARI_vs_paper=round(paper_ari(lab), 3)))
    return pd.DataFrame(rows)

summarize("unc")

> **Concept · the Adjusted Rand Index (ARI).** It compares two ways of grouping the same cells. ARI = 1: identical groupings; ARI ≈ 0: no more agreement than random labels. It does not require the clusters to have the same names or numbers, only to group the same cells. Here we compare *our* clusters with *the authors'* states (`ARI_vs_paper`).

In [ ]:
sc.pl.embedding(kera, basis="umap_uncorrected", color=["unc_r0.5", "sample"], ncols=2, wspace=0.3)
print(pd.crosstab(kera.obs["unc_r0.5"], kera.obs["sample"]))

### A4 · Batch correction *inside* the subset: Harmony on the timepoint

> **Concept** · In notebook 1 we had one donor and did not need any correction. Now, *within* the keratinocytes, the timepoint effect is strong enough to split states in two. **Harmony** (the method we met in notebook 1) moves the cells in PCA space so that each cluster contains cells of all samples, **without** touching the gene expression values. After Harmony we rebuild the graph and the UMAP from the corrected coordinates.
>
> Caution, this is a modelling choice with a cost: if a state *really* exists only at D7 (e.g. a wound-activated state), Harmony may partly merge it with its non-activated neighbours. That is why we will check afterwards that wound-related states are still visible (Part A6).

In [ ]:
ho = harmonypy.run_harmony(kera.obsm["X_pca"], kera.obs, ["sample"])
Z = np.asarray(ho.Z_corr)
if Z.shape[0] != kera.n_obs:          # some versions return the transpose
    Z = Z.T
kera.obsm["X_pca_harmony"] = Z

sc.pp.neighbors(kera, use_rep="X_pca_harmony", key_added="harmony", random_state=SEED)
sc.tl.umap(kera, neighbors_key="harmony", random_state=SEED)
kera.obsm["X_umap_harmony"] = kera.obsm["X_umap"].copy()
kera.obsm["X_umap"] = kera.obsm["X_umap_uncorrected"].copy()       # restore the default UMAP

for r in [0.3, 0.5, 0.8]:
    sc.tl.leiden(kera, neighbors_key="harmony", resolution=r, key_added=f"har_r{r}", flavor="igraph", directed=False, n_iterations=2, random_state=SEED)
pd.concat([summarize("unc"), summarize("har")], ignore_index=True)

In [ ]:
sc.pl.embedding(kera, basis="umap_harmony", color=["sample", "har_r0.5", "paper_state"], ncols=3, wspace=0.4)

> **Question** · The ARI vs the authors' states is higher after Harmony. Does it prove that Harmony 'found the truth'?
>
> <details><summary>Show answer</summary>
> No. It shows that, after correction, our grouping is closer to the grouping of the authors. But the authors also made choices (they integrated several donors with their own pipeline), so their labels are a reference, not a ground truth. A higher ARI is a reassuring sign, not a proof.
> </details>

### A5 · Choosing the resolution and describing the clusters

> **Concept · there is no 'right' resolution.** Criteria to decide, in order of importance: (1) **biological interpretability**: can I name each cluster from its marker genes? (2) **stability**: do the clusters persist when I change the resolution slightly? (3) **agreement with an independent reference** (the ARI above). We take **resolution 0.5** with Harmony: enough clusters to separate the known epidermal layers and the wound-related states, few enough to be interpretable.
>
> To name the clusters we use two complementary tools:
>
> - **Differential expression** (`rank_genes_groups`, Wilcoxon test): the genes higher in a cluster than in all others.
> - **Marker signatures**: for the known biology of the epidermis we define short gene lists (basal, proliferating, spinous, granular, wound-activated, hair follicle) and compute a **score** per cell (`score_genes`: mean expression of the list minus the mean of a random set of genes with similar expression). Scores are less noisy than a single gene.

In [ ]:
kera.obs["kcluster"] = kera.obs["har_r0.5"].astype(str).astype("category")
sc.tl.rank_genes_groups(kera, "kcluster", method="wilcoxon", pts=True, key_added="rank_genes_k")

signatures = {
    "Basal":           ["KRT15", "COL17A1", "ITGB4", "ITGA6", "DST"],
    "Proliferating":   ["MKI67", "TOP2A", "STMN1", "PTTG1", "HMGB2"],
    "Spinous":         ["KRT1", "KRT10", "DMKN", "KRTDAP", "PERP"],
    "Granular":        ["FLG", "LOR", "CALML5", "SPINK5", "SLURP1"],
    "Wound_activated": ["KRT6A", "KRT16", "KRT17", "S100A8", "S100A9"],
    "Hair_follicle":   ["SOX9"],
}
for name, genes in signatures.items():
    genes = [g for g in genes if g in kera.var_names]
    sc.tl.score_genes(kera, genes, score_name=f"score_{name}", random_state=SEED)

mean_scores = kera.obs.groupby("kcluster")[[f"score_{n}" for n in signatures]].mean()
mean_scores.columns = list(signatures)
sns.heatmap(mean_scores.loc[sorted(mean_scores.index, key=int)], cmap="vlag", center=0, annot=True, fmt=".2f")
plt.title("Mean signature score per cluster"); plt.show()

In [ ]:
sc.tl.dendrogram(kera, "kcluster", use_rep="X_pca_harmony")
sc.pl.dotplot(kera, var_names=[g for v in signatures.values() for g in v if g in kera.var_names],
              groupby="kcluster", dendrogram=True, standard_scale="var")
top = pd.concat({g: sc.get.rank_genes_groups_df(kera, group=g, key="rank_genes_k").head(8)["names"].reset_index(drop=True)
                 for g in sorted(kera.obs["kcluster"].cat.categories, key=int)}, axis=1)
top

### A6 · Naming the clusters

> **Concept** · We read the heatmap, the dot plot and the DE table together and give each cluster a name. Two cautions:
>
> - **Names are hypotheses.** Our names combine the layer (basal / spinous / granular), the activity (proliferating, wound-activated) and, when the genes do not tell a clear story, a descriptive label (`..._ribosomal_high`, `Stress_or_mixed`).
> - **Not every cluster is a cell type.** A cluster dominated by ribosomal and mitochondrial genes, or by heat-shock/stress genes, may be a technical state (dissociation stress, low-quality cells) rather than a biological one. We keep such clusters but we flag them.
>
> **Cluster numbers are not stable across computers.** Leiden uses random numbers and floating-point operations; two machines can give the *same groups* with *different numbers*. The solution below therefore does **not** use cluster numbers: it lists the marker genes of each state (from our reference run) and assigns each of *your* clusters to the state with which its top DE genes overlap most (one-to-one, Hungarian algorithm). If your clusters differ from ours, some may stay `unassigned`: then you name them by hand. Always check the assignment against your own dot plot.

In [ ]:
from scipy.optimize import linear_sum_assignment

# marker genes that characterise each state in the reference run (no cluster numbers here, on purpose)
state_markers = {
    "Basal":                     ["COL17A1", "KRT15", "DST", "CXCL14", "POSTN", "KRT5", "KRT14", "ITGB4"],
    "Basal_wound_activated":     ["S100A2", "ITGB1", "PFN1", "TNFRSF12A", "AREG", "ITGA3", "TNC"],
    "Hair_follicle_like":        ["ADGRL3", "SOX9", "C12orf75", "FOXC1", "BCAM"],
    "Proliferating":             ["H2AZ1", "HMGB2", "STMN1", "TUBA1B", "CKS1B", "TOP2A", "PTTG1", "MKI67"],
    "Spinous_early":             ["LGALS3", "KRT1", "KRT10", "LAMTOR4", "LGALS7", "LGALS7B", "GATA3"],
    "Spinous_late":              ["KRT2", "KRTDAP", "PERP", "DMKN", "CALML5", "SPINK5", "SCEL", "DSC1", "KLK11"],
    "Spinous_upper_S100A7+":     ["S100A7", "DBI", "S100A8", "CSTB", "RBP1", "TSPO", "PYCARD"],
    "Spinous_RP_high":           ["MT1X", "RPS8", "RPLP1", "RPL10A", "RPL17", "RPS19", "RPS24", "RPL13", "RPL32", "RPL6"],
    "Spinous_wound_activated_A": ["KRT6A", "S100A8", "KRT16", "S100A9", "LY6D", "FABP5", "SERPINB4", "S100A14"],
    "Spinous_wound_activated_B": ["KRT6A", "KRT16", "KRT6C", "KRT6B", "S100A9", "SFN", "IVL", "SERPINB1", "SPRR1B"],
    "Stress_or_mixed":           ["TIMP1", "HSPH1", "SGK1", "S100A6", "VIM", "S100A4", "SRGN", "EMP3"],
    "Granular":                  ["POF1B", "CNFN", "CST6", "SERPINA12", "SBSN", "NCCRP1", "KLK7", "CDSN", "SLURP1", "FLG"],
}

# how many of the 30 top DE genes of each of YOUR clusters belong to each marker list?
clusters = sorted(kera.obs["kcluster"].cat.categories, key=int)
top30 = {c: set(sc.get.rank_genes_groups_df(kera, group=c, key="rank_genes_k").head(30)["names"]) for c in clusters}
states = list(state_markers)
overlap = pd.DataFrame([[len(top30[c] & set(state_markers[s])) for c in clusters] for s in states], index=states, columns=clusters)

# one-to-one assignment cluster <-> state that maximises the total overlap (Hungarian algorithm)
rows, cols = linear_sum_assignment(-overlap.values)
assign = {clusters[j]: states[i] for i, j in zip(rows, cols) if overlap.iloc[i, j] >= 2}
for c in clusters:
    assign.setdefault(c, f"Cluster_{c}_unassigned")

plt.figure(figsize=(8, 4.5)); sns.heatmap(overlap, annot=True, cmap="Blues", cbar=False)
plt.xlabel("your cluster"); plt.title("shared genes between top-30 DE genes and the marker lists"); plt.show()
print({c: assign[c] for c in clusters})
if any("unassigned" in v for v in assign.values()):
    print("WARNING: some clusters were not recognised. Name them by hand from your dot plot and DE table.")

kera.obs["kstate"] = kera.obs["kcluster"].map(assign).astype("category")
sc.pl.embedding(kera, basis="umap_harmony", color="kstate", legend_loc="on data", legend_fontsize=7)

> **Question** · Two clusters contain mostly ribosomal genes (`RPL*`, `RPS*`). Biological state or artefact? How could you decide?
>
> <details><summary>Show answer</summary>
> Both are possible. Ribosomal-gene-high cells can be metabolically very active (real biology) or can be small/low-complexity cells where ribosomal transcripts dominate the profile (technical). To decide: check `n_genes` and `pct_mito` of these cells, check whether they come from one sample, check whether they co-express a layer marker (here they are spinous), and see whether the authors have an equivalent state. If it is only 'ribosomal high', we should not build a biological story on it.
> </details>

### A7 · Validation against the published annotation, and where the states live in time

> **Concept** · We cross-tabulate **our states** (rows) against **the authors' states** (columns). Normalizing each row to 100% shows, for each of our states, which published state its cells belong to. A good annotation gives rows with one dominant column; a split row means our cluster merges (or splits) published states.

In [ ]:
ct = pd.crosstab(kera.obs["kstate"], kera.obs["paper_state"], normalize="index")
plt.figure(figsize=(8, 4.5)); sns.heatmap(ct, cmap="Blues", annot=True, fmt=".2f", cbar_kws=dict(label="fraction of our state"))
plt.title(f"Our states vs published states (ARI = {paper_ari(kera.obs['kstate']):.2f})"); plt.show()

> **Concept · composition over time.** Now the biological payoff: the proportion of each state **in each timepoint**. In a wound, we expect basal/proliferating states to be present at D0 (normal skin), the wound-activated states to expand early (D1, D7) and to return to baseline at D30.
>
> Caution: proportions are **compositional** (they sum to 100%): if one state grows, the others shrink even if their absolute number is constant. And with one donor and one biopsy per timepoint, we have **no replicates**: we can describe, we cannot test.

In [ ]:
prop = pd.crosstab(kera.obs["sample"], kera.obs["kstate"], normalize="index") * 100
prop.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="tab20")
plt.ylabel("% of keratinocytes"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7); plt.xticks(rotation=0); plt.show()
prop.round(1)

> **Question** · State X represents 5% of the keratinocytes at D0 and 20% at D7. Can we say 'it is induced by the wound'?
>
> <details><summary>Show answer</summary>
> Only as a description. (1) Proportions are relative: X may have grown, or other states may have shrunk. (2) One biopsy per timepoint and one donor: no replicate, no statistical test, and a biopsy-to-biopsy difference in the site (e.g. more hair follicle) gives the same picture. (3) Batch effect between biopsies. The claim becomes solid with several donors and a differential abundance test.
> </details>

**Checkpoint.** We save the keratinocytes with their sub-states: Part B and C start from this file.

In [ ]:
kera.write_h5ad(f"{OUT}/keratinocytes_substates.h5ad")
print("saved:", f"{OUT}/keratinocytes_substates.h5ad")

> **Take-home (Part A)** · Zoom in = redo the pipeline on the subset from raw counts. Resolution is a choice judged by interpretability, stability and an external reference. Inside a subset the batch effect (here the timepoint) can split real states in two: check the purity and correct if needed. Names are hypotheses.

## Part B · Trajectory and pseudotime

**Goal.** Test whether the keratinocyte states form an ordered path (basal → spinous → granular), assign every cell a position along it, and find the genes that change along the path.

**To discuss together:**

- Is differentiation a set of discrete boxes or a continuum? What would each view predict in the data?
- A trajectory is computed from a snapshot. What does 'time' mean here?
- What should we fix before computing a pseudotime, and who decides it: the algorithm or us?

### B1 · The idea of pseudotime

> **Concept** · Each cell is a **snapshot**: we destroyed it to read its RNA. But in a tissue that is renewing itself, cells are at *different moments* of the same process (a basal cell, a cell leaving the basal layer, a cell about to cornify...). If the process is continuous, the cells form a **line (or tree) in gene-expression space**, and we can order them along it. The position along that line is the **pseudotime**.
>
> Three crucial points:
>
> 1. **Pseudotime is not real time.** It is a measure of *transcriptional progress* along a path. Here it will represent *epidermal differentiation*, not the days after wounding (D0 → D30).
> 2. **The direction is not computed, it is decided.** The algorithm knows distances between cells, not which end is the 'start'. We must choose a **root cell** from biological knowledge (basal cells are the stem-like layer).
> 3. **A trajectory always comes out.** Give the algorithm any cloud of points and it will draw a path through it. Only biology tells us whether it is meaningful.
>
> We use two tools: **PAGA** (a coarse "map" of which states are connected) and **diffusion pseudotime, DPT** (a number per cell, computed from random walks on the kNN graph).

### B2 · PAGA: a map of connectivity between states

> **Concept** · PAGA (*partition-based graph abstraction*) takes the clusters (here our states) and asks, for each pair, *how many kNN edges connect their cells compared with what is expected by chance?* The result is a small graph: **nodes = states**, **edges = connectivity** (thicker = stronger). It is robust to the noisy detail of UMAP and it distinguishes a continuum (connected chain) from truly separate populations (disconnected nodes).
>
> We compute it on the **Harmony graph** (`neighbors_key="harmony"`), consistently with how we defined the states.

In [ ]:
sc.tl.paga(kera, groups="kstate", neighbors_key="harmony")
sc.pl.paga(kera, layout="fr", random_state=SEED, threshold=0.05, node_size_scale=2, edge_width_scale=0.6, fontsize=7)

> **Question** · A state is connected to nothing in the PAGA graph. What are the possible interpretations?
>
> <details><summary>Show answer</summary>
> (1) It is a genuinely distinct population (different lineage, e.g. hair follicle vs interfollicular epidermis); (2) it is a technical group (stressed/low-quality cells) which does not sit on the biological axis; (3) the threshold was too high and hid weak edges. Lowering the threshold and looking at the underlying numbers (`kera.uns['paga']['connectivities']`) tells (3) from the rest.
> </details>

### B3 · Choosing the root and computing diffusion pseudotime

> **Concept · diffusion pseudotime (DPT).** From the root cell, imagine a random walker that moves on the kNN graph, preferring close neighbours. The time the walker needs to reach each cell, averaged over all possible walk lengths, is that cell's pseudotime: close to the root = small, far = large. Technically scanpy first computes a **diffusion map** (a non-linear low-dimensional embedding where distances reflect walk probabilities) and then DPT on it.
>
> **The root.** We pick it in a data-driven but biologically justified way: the state with the highest **basal signature score**, and inside it the cell with the highest score. `kera.uns["iroot"]` must be the **integer position** of that cell in the object (not its name).

In [ ]:
basal = kera.obs["score_Basal"].values
root_cluster = kera.obs.groupby("kcluster")["score_Basal"].mean().idxmax()
cand = np.where(kera.obs["kcluster"].values == root_cluster)[0]
kera.uns["iroot"] = int(cand[np.argmax(basal[cand])])
print("root state:", kera.obs["kstate"].iloc[kera.uns["iroot"]], "| root cell:", kera.obs_names[kera.uns["iroot"]])

sc.tl.diffmap(kera, neighbors_key="harmony")
sc.tl.dpt(kera, neighbors_key="harmony")
kera.obs["pt_basal_root"] = kera.obs["dpt_pseudotime"].copy()
print(kera.obs["dpt_pseudotime"].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.embedding(kera, basis="umap_harmony", color="dpt_pseudotime", cmap="viridis", ax=axes[0], show=False, title="pseudotime")
sc.pl.embedding(kera, basis="umap_harmony", color="kstate", ax=axes[1], show=False, legend_loc="on data", legend_fontsize=6, title="state")
sc.pl.embedding(kera, basis="umap_harmony", color="sample", ax=axes[2], show=False, title="timepoint")
plt.tight_layout(); plt.show()

ok = np.isfinite(kera.obs["dpt_pseudotime"])
order = kera.obs[ok].groupby("kstate")["dpt_pseudotime"].median().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=kera.obs[ok], x="dpt_pseudotime", y="kstate", order=order, ax=axes[0], fliersize=0)
sns.boxplot(data=kera.obs[ok], x="dpt_pseudotime", y="sample", ax=axes[1], fliersize=0)
plt.tight_layout(); plt.show()

> **Question** · The median pseudotime is almost the same at D0, D1, D7 and D30. Is the method broken?
>
> <details><summary>Show answer</summary>
> No: it is what we should expect. Pseudotime here measures progress along *epidermal differentiation* (basal to granular), a process that is going on at every moment, in healthy and wounded skin. It is orthogonal to the experimental time. A different trajectory (for example *non-activated → wound-activated*) could correlate with the timepoint; we would need to choose the root and the path for that question.
> </details>

### B4 · Break it: what if we choose another root?

> **Break it** · The algorithm does not know the biology. We now put the root on the most **granular** cell (the terminal layer) and recompute. If the data really form a basal-to-granular line, the new pseudotime should be the *mirror image* of the first one (strongly negative rank correlation). Different results would reveal that the structure is not a simple line.

In [ ]:
root_basal = kera.uns["iroot"]
gran = kera.obs["score_Granular"].values
g_cl = kera.obs.groupby("kcluster")["score_Granular"].mean().idxmax()
cand_g = np.where(kera.obs["kcluster"].values == g_cl)[0]
kera.uns["iroot"] = int(cand_g[np.argmax(gran[cand_g])])
sc.tl.dpt(kera, neighbors_key="harmony")
kera.obs["pt_granular_root"] = kera.obs["dpt_pseudotime"].copy()

kera.uns["iroot"] = root_basal                                   # restore everything
kera.obs["dpt_pseudotime"] = kera.obs["pt_basal_root"].copy()

ok = np.isfinite(kera.obs["pt_basal_root"]) & np.isfinite(kera.obs["pt_granular_root"])
rho, _ = spearmanr(kera.obs.loc[ok, "pt_basal_root"], kera.obs.loc[ok, "pt_granular_root"])
print(f"Spearman correlation between the two pseudotimes: {rho:.2f}")
plt.figure(figsize=(4.5, 4)); plt.scatter(kera.obs.loc[ok, "pt_basal_root"], kera.obs.loc[ok, "pt_granular_root"], s=2, alpha=0.3)
plt.xlabel("pseudotime from a basal root"); plt.ylabel("pseudotime from a granular root"); plt.show()

### B5 · Which genes change along the path?

> **Concept** · The point of pseudotime is to find **genes whose expression varies along it**. A simple, robust approach: for each gene compute the **Spearman correlation** between its expression and the pseudotime (rank-based, so insensitive to the exact scale and to outliers). Positive = increases along the path (late/differentiated), negative = decreases (early/basal).
>
> To avoid noise we keep only the genes detected in at least 10% of the cells. Then we draw the top genes as a **heatmap over pseudotime**: cells are sorted by pseudotime, grouped into 20 equal-size bins, the mean expression per bin is computed and z-scored per gene.
>
> Caution: **the p-values of such a test are meaningless here.** The pseudotime was computed *from* the expression of the same genes, so we are testing the genes against a quantity built from them (circularity), and 11 000 cells are not 11 000 independent samples. Use the ranking, not the p-value.

In [ ]:
pt = kera.obs["dpt_pseudotime"].values
ok = np.isfinite(pt)
frac = np.asarray((kera.X > 0).mean(axis=0)).ravel()
keep = frac >= 0.10
Xd = pd.DataFrame(kera[ok][:, keep].X.toarray(), columns=kera.var_names[keep])

rho = Xd.rank().corrwith(pd.Series(pt[ok]).rank())               # Spearman = Pearson on ranks
rho = rho.sort_values()
print("most decreasing:", list(rho.index[:12]))
print("most increasing:", list(rho.index[-12:][::-1]))

In [ ]:
sel = list(rho.index[:15]) + list(rho.index[-15:])
bins = pd.qcut(pd.Series(pt[ok]).rank(method="first"), 20, labels=False)
M = Xd[sel].groupby(bins.values).mean()
Mz = (M - M.mean()) / M.std()
plt.figure(figsize=(9, 7)); plt.rcParams["ytick.labelsize"] = 8; sns.heatmap(Mz.T, cmap="vlag", center=0, xticklabels=False, yticklabels=True, cbar_kws=dict(label="z-score"))
plt.xlabel("pseudotime (20 bins, early → late)"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, g in zip(axes, ["KRT15", "KRT10", "KRT2", "CALML5"]):
    if g in kera.var_names:
        sc.pl.embedding(kera, basis="umap_harmony", color=g, ax=ax, show=False, cmap="Reds", title=g)
plt.tight_layout(); plt.show()

> **Question** · A gene correlates strongly with pseudotime. Is it a *driver* of differentiation?
>
> <details><summary>Show answer</summary>
> Not necessarily. Correlation is not causation, and pseudotime is built from expression, so many genes just *follow* the process (they are consequences of the cell's identity). Evidence for a driver would need regulatory knowledge (transcription factors, motif analysis, perturbation experiments).
> </details>

> **Take-home (Part B)** · Pseudotime orders cells along a *chosen* direction on a graph. The root is a biological decision, the path must be validated by marker genes, and pseudotime is not experimental time. Use correlations to rank candidate genes, not to claim significance.

## Part C · Gene set enrichment

**Goal.** Turn a list of genes (or a ranking of all genes) into biological processes, using two complementary methods, and learn when each is appropriate.

**To discuss together:**

- A list of 150 genes is not a result. How do we get from genes to a biological statement?
- What is the 'universe' (background) of a test, and why does it change the answer?
- Why can the same data give different answers with a gene list (ORA) and a ranking (GSEA)?

### C1 · Gene sets: where do they come from?

> **Concept** · A **gene set** is a named list of genes that share a property: a biological process (Gene Ontology, **GO**), a pathway (Reactome, KEGG), a cell-type signature, a chromosomal location... A **library** is a collection of such sets. Libraries are curated by humans and by algorithms: they reflect what was *studied*, not what exists. Well-studied processes (cell cycle, inflammation) have large, rich sets; poorly studied tissues are under-represented. Always ask which library you used and which version.
>
> We use **GO Biological Process 2023** as distributed by Enrichr, through `gseapy`. A library is downloaded once and then cached as a local JSON file in `data/gene_sets`: this way the practical works again without internet and the result is reproducible (the library version does not change under our feet).

In [ ]:
def load_gene_sets(name="GO_Biological_Process_2023"):
    path = f"{GENESET_DIR}/{name}.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    lib = gp.get_library(name=name, organism="Human")          # needs internet, only the first time
    with open(path, "w") as f:
        json.dump(lib, f)
    return lib

gene_sets = load_gene_sets()
sizes = pd.Series({k: len(v) for k, v in gene_sets.items()})
print(len(gene_sets), "gene sets; median size", int(sizes.median()), "genes")
print(sizes.sort_values().tail(3).to_dict())

> **Question** · A GO term contains 2 000 genes, another 8. Which one is easier to find 'significant', and which one is more informative?
>
> <details><summary>Show answer</summary>
> The big one is easy to hit by chance and tells little ("metabolic process"). The tiny one is specific but a single gene can drive the result. This is why we filter the sizes (e.g. 15-500 genes) and why enrichment results always need a critical reading of the terms.
> </details>

### C2 · Method 1 · Over-representation analysis (ORA)

> **Concept** · ORA starts from a **list of genes** (e.g. the genes up-regulated in a cluster) and asks, for each gene set: *is the overlap between my list and the set larger than expected by chance?* It is a **hypergeometric test** (equivalently Fisher's exact test): drawing 150 genes at random from a universe of N genes, how likely is it to get k or more from a set of size K?
>
> Three decisions determine the result:
>
> 1. **The list.** Here: the top 150 genes with adjusted p < 0.05 and log fold change > 1 in the *wound-activated* keratinocyte cluster (the one with the highest `Wound_activated` score), versus all the other keratinocytes. A threshold is arbitrary: a different cut-off gives a different list.
> 2. **The universe (background).** Only genes that *could* have been in the list should count. In a single-cell experiment those are the genes that are expressed in our cells. A default background of "all genes in the library" would make *any* expressed-in-skin gene look enriched. We use the genes detected in our keratinocytes.
> 3. **Multiple testing.** We test thousands of sets: we use the **adjusted p-value** (Benjamini-Hochberg FDR), never the raw one.

In [ ]:
wound_cluster = kera.obs.groupby("kcluster")["score_Wound_activated"].mean().idxmax()
print("target cluster:", wound_cluster, "=", kera.obs.loc[kera.obs["kcluster"] == wound_cluster, "kstate"].iloc[0])

de = sc.get.rank_genes_groups_df(kera, group=wound_cluster, key="rank_genes_k")
up = de[(de["pvals_adj"] < 0.05) & (de["logfoldchanges"] > 1)].head(150)["names"].tolist()
background = kera.var_names.tolist()
print(len(up), "genes in the list; universe:", len(background), "genes")

enr = gp.enrichr(gene_list=up, gene_sets=gene_sets, background=background, outdir=None, no_plot=True)
ora = enr.results.sort_values("Adjusted P-value")
ora[["Term", "Overlap", "Adjusted P-value", "Genes"]].head(10)

In [ ]:
top = ora.head(10).iloc[::-1]
plt.figure(figsize=(8, 4)); plt.barh(top["Term"], -np.log10(top["Adjusted P-value"]), color="tab:red")
plt.xlabel("-log10 adjusted p-value"); plt.title(f"ORA · up in {kera.obs.loc[kera.obs['kcluster'] == wound_cluster, 'kstate'].iloc[0]}"); plt.tight_layout(); plt.show()

### C3 · Break it: change the universe

> **Break it** · We repeat the test **without** declaring the background. `gseapy` then uses, as the universe, the genes that appear in the gene set library. Compare the adjusted p-values and the order of the terms with the previous table. This is a very common mistake (it is the default in many web tools), and it silently changes the p-values (and sometimes the order of the terms), because the test is run against a universe that does not describe our experiment.

In [ ]:
enr_nobg = gp.enrichr(gene_list=up, gene_sets=gene_sets, outdir=None, no_plot=True)
cmp = ora.set_index("Term")[["Adjusted P-value"]].join(enr_nobg.results.set_index("Term")[["Adjusted P-value"]], lsuffix="_background", rsuffix="_default")
cmp.sort_values("Adjusted P-value_background").head(8)

> **Question** · The p-values change when we change the universe. Which ones do we believe?
>
> <details><summary>Show answer</summary>
> The ones computed with the universe that describes our experiment: the genes that could have been selected, i.e. those detectable in our cells. With another universe the test answers a different question, so a smaller (or larger) p-value obtained with the wrong universe is a bias, not a weaker or stronger result. In practice one may further restrict the universe to genes that are annotated in the library; the important point is to decide it consciously.
> </details>

### C4 · Method 2 · GSEA on a ranked list (preranked)

> **Concept** · ORA throws away information: it cuts the genes into 'in the list' / 'not in the list' and ignores how strong the change is. **GSEA** (gene set enrichment analysis) avoids the cut-off. It starts from **all genes ranked** by a statistic (here the Wilcoxon score of the comparison) and walks down the list, increasing a running sum when we meet a gene of the set and decreasing it otherwise. The maximum deviation is the **enrichment score (ES)**; it is normalised for set size (**NES**) and its significance is estimated by **permutations** (here of the gene labels, 200 for speed; use 1 000+ in a real analysis).
>
> NES > 0: the set is concentrated at the *top* of the ranking (up-regulated); NES < 0: at the *bottom* (down-regulated).
>
> The question we ask now is different from C2: **how does the whole keratinocyte population change 1 day after wounding (D1) compared with healthy skin (D0)?** The ranking is from `rank_genes_groups(sample, D1 vs D0)`.
>
> **Statistical caveat (important).** One donor, one biopsy per timepoint: the cells of D1 are not independent replicates of "wounded skin", they are 4 795 cells of the same biopsy. Differences between *biopsies* (position, depth, handling) are mixed with the effect of the wound. The ranking and the pathways are a hypothesis-generating description; confirming them would require several donors and a **pseudobulk** analysis (summing the counts per donor and timepoint).

In [ ]:
sc.tl.rank_genes_groups(kera, "sample", groups=["D1"], reference="D0", method="wilcoxon", key_added="rank_genes_d1")
d1 = sc.get.rank_genes_groups_df(kera, group="D1", key="rank_genes_d1")
rnk = d1.set_index("names")["scores"].sort_values(ascending=False)
print(len(rnk), "ranked genes; top:", rnk.head(5).index.tolist(), "| bottom:", rnk.tail(5).index.tolist())

pre = gp.prerank(rnk=rnk, gene_sets=gene_sets, min_size=15, max_size=500, permutation_num=200,
                 seed=SEED, threads=2, outdir=None, no_plot=True, verbose=False)
gsea = pre.res2d.sort_values("NES", ascending=False)
gsea[["Term", "NES", "NOM p-val", "FDR q-val", "Lead_genes"]].head(8)

In [ ]:
sel = pd.concat([gsea.head(8), gsea.tail(8)]).iloc[::-1]
plt.figure(figsize=(8, 5))
plt.barh(sel["Term"], sel["NES"].astype(float), color=np.where(sel["NES"].astype(float) > 0, "tab:red", "tab:blue"))
plt.xlabel("NES (normalised enrichment score)"); plt.tight_layout(); plt.show()

term = gsea.iloc[0]["Term"]
gp.gseaplot(rank_metric=pre.ranking, term=term, **pre.results[term])

> **Question** · ORA on the 'wound-activated' cluster and GSEA on D1 vs D0 both return wound/keratinization terms. Do they have to agree?
>
> <details><summary>Show answer</summary>
> Not necessarily, and where they agree the confidence goes up. They answer different questions: ORA looks at a *cluster* (a sub-population) with a threshold, GSEA at the *whole population* between two conditions, using the full ranking. A pathway driven by 8% of the cells (the wound-activated ones) can be strong in ORA and weak in GSEA, and the reverse for a small change spread over all cells.
> </details>

> **Question** · Why do pathway names (GO terms) often look similar to each other in the top of the table?
>
> <details><summary>Show answer</summary>
> GO is hierarchical and sets overlap heavily ("keratinocyte differentiation", "epidermis development", "cornification" share many genes). Ten significant terms may be one single biological signal. Group the terms by shared genes before telling a story with ten findings.
> </details>

> **Take-home (Part C)** · A gene set result is only as good as (i) the list or ranking, (ii) the universe and (iii) the library. ORA needs a list and a correct background; GSEA uses a ranking and avoids cut-offs. With one donor, pathways are descriptions, not tests of a biological hypothesis.

## Part D · Take-home messages and limits

| Method | What it gives | What can go wrong |
|---|---|---|
| **Sub-clustering** | sub-populations inside a cell type | batch effects split real states; arbitrary resolution; technical clusters (ribosomal, stress) mistaken for biology |
| **PAGA + pseudotime** | ordering of cells along a continuous process | the root and the direction are *our* assumption; pseudotime is not real time; a path always comes out |
| **Gene trends** | genes that follow the path | circularity (the path is built from the genes); correlation is not causation |
| **ORA** | processes over-represented in a list | arbitrary cut-off, wrong universe, redundant terms |
| **GSEA** | processes shifted in a full ranking | permutation p-values with correlated cells; the ranking comes from pseudo-replicates |

**What we would do next with more data:** several donors (to separate biology from biopsy-to-biopsy variation), a **pseudobulk** differential expression per donor, a **differential abundance** test (e.g. Milo) for the proportions of the states, **RNA velocity** for the direction of the trajectory, and the **spatial** data of the same study to see *where* the wound-activated states are in the tissue.